# Session 4 - Optimization and Reporting

Goal: optimize prompt strategy, auto-generate a scientific report with AI, and produce interview-ready final deliverables.

## Learning Objectives

1. Evaluate multiple prompt variants systematically.
2. Apply a Planner-Executor-Critic-Human Gate agent architecture to self-optimize prompt choice.
3. Use an LLM to auto-generate a scientific analysis report.
4. Build concise report and resume bullets.

## TODO Mapping to Source Files

- src/optimization/prompt_optimizer.py
- src/optimization/bandit_optimizer.py
- src/optimization/agent_loop.py
- src/agents/analyst_agent.py
- src/agents/human_gate.py
- src/reporting/summarizer.py
- src/reporting/ai_reporter.py
- src/common/logging_utils.py

In [ ]:
# Path setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_ROOT = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_ROOT))

print("Project root:", PROJECT_ROOT)

---
## 1. Priming Effect and AI Development

**Why does syntactic priming matter for AI?**

The priming effect shows that LLM outputs can be influenced by the structure of the input prompt. This has important implications:

| Area | Implication |
|------|-------------|
| **AI Alignment** | If we can control output structure via priming, we can steer LLMs toward safer, more predictable responses |
| **Robustness** | Priming sensitivity means LLMs may be vulnerable to adversarial prompts that manipulate output style |
| **Controllability** | Understanding priming helps design better prompt engineering strategies |
| **Evaluation** | Priming rate is a measurable metric for comparing LLM behavior across models and prompts |

**Discussion**: How might syntactic priming relate to prompt injection attacks? Can priming be used defensively?

---
## 2. Effective Experiment Design

**Key principles for multi-condition experiments**:

1. **Control variables**: Keep temperature, max_tokens, model name constant across conditions
2. **Randomization**: Randomize trial order to avoid order effects
3. **Block design**: Group trials by condition to control for time-based confounds
4. **Latin square**: Rotate condition order across subjects to balance carryover effects
5. **Power analysis**: Ensure enough trials to detect your expected effect size

**Your task**: Implement `generate_prompt_variants()` in `src/optimization/prompt_optimizer.py`.
Consider variants like:
- `minimal` (prime only)
- `instructed` (prime + explicit syntax instruction)
- `few_shot` (prime + example responses)

---
## 3. Modern Multi-Agent Optimization Architecture

Recall your Session 3 A/B test: it compares two prompt strategies once and makes a single
decision. A more modern setup treats optimization as a small team of agents with distinct jobs,
looping until a human signs off:

```mermaid
flowchart LR
    P[Planner: bandit_optimizer] --> E[Executor: existing runner / dialogue_runner]
    E --> C[Critic: analyst_agent]
    C --> P
    C --> H[Human Gate]
    H --> P
```

- **Planner** picks which prompt variant to try next.
- **Executor** runs it (reuses your Session 2 `runner`/`dialogue_runner` — do not rebuild this).
- **Critic** decides if the prime effect is real (reuses your Session 3 `metrics`/`stats` — do not recompute stats here).
- **Human Gate** approves before anything is called "the winner."

This separation of planning / execution / evaluation / oversight is the core pattern behind most
modern agentic systems, regardless of how fancy the individual agents are.


### What Actually Gets Tuned in This Agentic Loop?

Every agent in the diagram above has at least one parameter worth tuning. Nothing here needs a
new file — these are arguments you'll pass into the functions you already planned to implement:

| Parameter | Lives in | What it controls |
|---|---|---|
| `epsilon` | `bandit_optimizer` | How often the Planner explores a new variant vs. exploits the current best |
| `num_rounds` | `agent_loop` | Total optimization budget (more rounds = more confidence, more cost) |
| reward metric | `agent_loop` (choice of what to pass to `update_variant_reward`) | Whether "success" means priming rate, effect size, or a cost-adjusted score |
| `minimum_effect` | `ab_test.recommend_rollout` | How big an improvement must be before rollout is recommended |
| confidence threshold | `analyst_agent` | When the Critic calls a result "significant" vs. "inconclusive" |
| `num_trials` per round | `experiments.config` | Sample size per variant — affects statistical power |
| `auto_approve` | `human_gate` | Whether the Human Gate blocks for real input or auto-approves (e.g. for CI re-runs) |

**Reflection**: If you only had budget for 20 total trials, would you spend them on more rounds
with fewer trials each, or fewer rounds with more trials each? What does that trade off?


### Beyond This Project: What Gets Tuned in Multi-Agent Systems Generally

The table above covers this project's loop. In multi-agent LLM systems more broadly, optimization
happens at several layers — `temperature` is only one knob at the first layer:

| Layer | Examples |
|---|---|
| **Generation params** (per agent) | `temperature`, `top_p`/`top_k`, `max_tokens`, model choice per role (cheap "router" model vs. strong "reasoner" model) |
| **Prompt-level** | Prompt wording, few-shot example selection, system instructions — searched automatically via APE, DSPy-style compilers, evolutionary search, or bandits (like this project) instead of hand-tuning |
| **Architecture / orchestration** | Agent topology (sequential, parallel fan-out, hierarchical manager-worker, debate/voting), routing logic, number of specialized agents |
| **Memory & context** | What history each agent sees, RAG retrieval `top_k`/chunking, memory summarization strategy |
| **Tool-use** | Which tools each agent can call, tool-selection strategy, retry/fallback on tool failure |
| **Control flow** | Max iterations before forced stop, early-stopping/consensus conditions, per-step timeout and retry policy |
| **Evaluation-driven optimization** | Reward/scoring definition, then a search strategy over that reward: one-shot A/B test, adaptive bandit (this project), self-consistency/majority voting, full RLHF (fine-tunes the model itself) |
| **Cost & latency** | Caching, batching, model tiering — in production, cost x latency x number of agent hops is often the binding constraint, not raw accuracy |

**Reflection**: Of the layers you *didn't* touch in this project (architecture, memory, tools,
cost), which would matter most if you scaled this from a research prototype to a production system?


### Auto-Analysis Agent: does the prime affect the output?

Right now, deciding "is this effect real?" is a manual step you do by reading chi-square/effect-size
numbers. An **auto-analysis agent** turns that judgment call into a reusable function so any
experiment run can be auto-triaged.

**Your task**: Implement `auto_analyze_prime_effect()` in `src/agents/analyst_agent.py`.
It should read the `metrics_summary`/`stats_summary` dicts you already built in Session 3 —
not recompute chi-square or effect size itself — and return a verdict.

**Reflection**: What confidence threshold should flip the verdict from `"inconclusive"` to
`"significant"`? Should that threshold depend on sample size?

In [ ]:
# --- 3.5a Auto-Analysis Agent ---
# Implement this in src/agents/analyst_agent.py
#
# Hint: this function is an interpreter, not a calculator — it consumes results
# you already computed in Session 3 (analysis/metrics.py, analysis/stats.py).
#
# from agents.analyst_agent import auto_analyze_prime_effect
# verdict = auto_analyze_prime_effect(metrics_summary, stats_summary)
# print(verdict["verdict"], verdict["confidence"], verdict["reasoning"])

# TODO: Implement auto_analyze_prime_effect(metrics_summary, stats_summary) -> dict
# TODO: Implement summarize_verdict_for_human(verdict) -> str

### Self-Optimizing Loop (Lightweight Reinforcement Learning)

Instead of testing one candidate prompt once, we can let an agent keep trying variants and
learn which one works best over several rounds. The simplest way to do this without any ML
background or library is an **epsilon-greedy multi-armed bandit**:

- Each prompt variant is an "arm."
- The **reward** for a round is a number like the priming rate or effect size from that round.
- With probability `epsilon`, **explore**: try a random variant.
- Otherwise, **exploit**: try the variant with the best average reward so far.
- After each round, update that variant's running average reward.

This is real reinforcement learning (a bandit is RL without state transitions), just scaled down
to something you can trace by hand.

**Your task**: Implement `src/optimization/bandit_optimizer.py`
(`initialize_variant_stats`, `select_variant_epsilon_greedy`, `update_variant_reward`)
and `run_self_optimizing_loop()` in `src/optimization/agent_loop.py`, which should only
orchestrate calls to your existing Session 2/3 functions plus the bandit.

In [ ]:
# --- 3.5b Self-Optimizing Bandit Loop ---
# Implement bandit functions in src/optimization/bandit_optimizer.py
# Implement the orchestrator in src/optimization/agent_loop.py
#
# Hint: variant_stats shape is {variant_name: {"trials": int, "total_reward": float, "avg_reward": float}}
#
# from optimization.agent_loop import run_self_optimizing_loop
# result = run_self_optimizing_loop(base_prompt_style="minimal", num_rounds=5, epsilon=0.2)
# print(result["best_variant"], result["history"])

# TODO: Implement initialize_variant_stats(variants) -> dict
# TODO: Implement select_variant_epsilon_greedy(variant_stats, epsilon, seed) -> str
# TODO: Implement update_variant_reward(variant_stats, variant_name, reward) -> dict
# TODO: Implement run_self_optimizing_loop(base_prompt_style, num_rounds, epsilon, auto_approve) -> dict

### Human-in-the-Loop Rollout Gate

An agent that can pick a "winner" on its own is convenient, but a fully autonomous rollout is
risky — the bandit's best variant might be a statistical fluke, or might optimize a metric that
doesn't reflect real quality. A **human gate** adds a required checkpoint before anything ships.

**Your task**: Implement `request_human_approval()` in `src/agents/human_gate.py` and
`finalize_rollout_decision()` in `src/optimization/agent_loop.py`. In the notebook, call it with
`auto_approve=False` so it prompts you interactively; set `auto_approve=True` only for
non-interactive re-runs.

In [ ]:
# --- 3.5c Human-in-the-Loop Gate ---
# Implement this in src/agents/human_gate.py and src/optimization/agent_loop.py
#
# from optimization.agent_loop import finalize_rollout_decision
# decision = finalize_rollout_decision(variant_stats, best_variant="instructed", auto_approve=False)
# print(decision["approved"], decision["notes"])

# TODO: Implement request_human_approval(proposal, auto_approve) -> dict
# TODO: Implement log_human_decision(decision, log_path) -> None
# TODO: Implement finalize_rollout_decision(variant_stats, best_variant, auto_approve) -> dict

### Reflection

A one-shot A/B test looks at the data once and decides. The adaptive bandit loop looks at the
data repeatedly and adapts as it goes. What statistical risk does "peeking" at results
mid-experiment introduce, and how would you mitigate it (e.g. pre-registered stopping rules,
correcting p-values for repeated looks)?

### Optional Preview: The Same Loop in a Modern Agent Framework

Everything above is plain Python on purpose — no framework required to understand Planner /
Executor / Critic / Human Gate. But in industry you'll often see this same loop expressed with
an agent-orchestration framework. Of the common options (LangGraph, CrewAI, AutoGen), **LangGraph**
needs the least code for a loop like ours, because our diagram *is* already a graph: nodes are
the Planner/Executor/Critic/Human Gate functions you're implementing, and edges are the arrows in
the mermaid diagram from Section 3.5.

This cell is **illustrative only** — nothing here is wired into `src/`, nothing runs as-is, and
`langgraph` is not a dependency you need to install to complete this project. It's here so you can
recognize the pattern if you see it in a job description or an existing codebase.

In [ ]:
# --- Illustrative only: NOT part of the assignment, does not run as-is ---
#
# pip install langgraph
#
# from langgraph.graph import StateGraph, END
# from typing import TypedDict
#
# class OptimizationState(TypedDict):
#     variant_stats: dict
#     best_variant: str
#     verdict: dict
#     approved: bool
#
# # Each node below is just a thin wrapper around the SAME functions you already
# # implement in bandit_optimizer.py / agent_loop.py / analyst_agent.py / human_gate.py.
#
# def planner_node(state: OptimizationState) -> OptimizationState:
#     variant = select_variant_epsilon_greedy(state["variant_stats"], epsilon=0.2)
#     state["best_variant"] = variant
#     return state
#
# def executor_node(state: OptimizationState) -> OptimizationState:
#     # run_trial_batch(...) / execute_experiment(...) from Session 2 goes here
#     return state
#
# def critic_node(state: OptimizationState) -> OptimizationState:
#     state["verdict"] = auto_analyze_prime_effect(metrics_summary, stats_summary)
#     return state
#
# def human_gate_node(state: OptimizationState) -> OptimizationState:
#     decision = request_human_approval({"variant": state["best_variant"]})
#     state["approved"] = decision["approved"]
#     return state
#
# graph = StateGraph(OptimizationState)
# graph.add_node("planner", planner_node)
# graph.add_node("executor", executor_node)
# graph.add_node("critic", critic_node)
# graph.add_node("human_gate", human_gate_node)
#
# graph.set_entry_point("planner")
# graph.add_edge("planner", "executor")
# graph.add_edge("executor", "critic")
# graph.add_edge("critic", "human_gate")
# # Loop back to the planner until a human approves, then stop.
# graph.add_conditional_edges("human_gate", lambda s: END if s["approved"] else "planner")
#
# app = graph.compile()
# final_state = app.invoke({"variant_stats": {}, "best_variant": "", "verdict": {}, "approved": False})

# Compare this to your plain-Python run_self_optimizing_loop(): same 4 nodes, same
# arrows as the Section 3.5 diagram — the framework just gives you a `.invoke()` runner,
# built-in state passing, and (in real use) visualization/checkpointing for free.

---
## 4. AI-Generated Analysis Report

Use an LLM to automatically write a scientific summary of your findings.
This is implemented in `src/reporting/ai_reporter.py`.

**How it works**:
1. Build a structured text block with experiment config, metrics, and stats
2. Pass it to an LLM with a prompt like: "Write a 3-paragraph scientific summary..."
3. The LLM returns a structured report with Abstract, Methods, Results, Discussion

**Exercise**: Compare the AI-generated report with your own manual interpretation.
When is AI-generated analysis trustworthy? When does it hallucinate?

```python
def build_report_context(experiment_summary, metrics, stats):
    # Combine all results into a formatted string
    pass

def generate_ai_report(context, llm_func):
    # Call LLM with context and return structured report
    pass
```

In [ ]:
# --- 4. AI-Generated Report ---
# Implement this in src/reporting/ai_reporter.py -> generate_ai_report()
#
# Hint: Build a context string with all experiment results, then call an LLM.
#
# def build_report_context(experiment_summary, metrics, stats):
#     lines = []
#     lines.append("=== EXPERIMENT CONFIG ===")
#     for k, v in experiment_summary.items():
#         lines.append(f"  {k}: {v}")
#     lines.append("\n=== PRIMING RATES ===")
#     for cond, rate in sorted(metrics.get('priming_rates', {}).items()):
#         lines.append(f"  {cond}: {rate:.1%}")
#     return "\n".join(lines)
#
# def generate_ai_report(context, llm_func):
#     prompt = f"Write a scientific summary...\n\n{context}"
#     return llm_func(prompt)

# TODO: Implement build_report_context() and generate_ai_report()
# TODO: Compare AI report with your own manual interpretation

---
## 5. Final Report and Resume

**Your task**: Implement `build_project_summary()`, `render_resume_bullets()`, and `export_markdown_report()` in `src/reporting/summarizer.py`.

### Interview Narrative Checklist

When presenting this project in an interview, be ready to discuss:
1. **Problem**: How can we measure and control LLM output structure?
2. **Method**: Multi-agent experiment loop with controlled stimuli
3. **Metrics**: Priming rate, effect size, chi-square, logistic regression
4. **Optimization**: A/B testing of prompt strategies
5. **Results**: Key findings and their implications for AI alignment
6. **Limitations**: What would you improve with more time?

In [ ]:
# --- 5. Final Report ---
# Implement this in src/reporting/summarizer.py
#
# TODO: Implement build_project_summary(experiment, analysis, optimization) -> dict
# TODO: Implement render_resume_bullets(summary, max_bullets=4) -> list[str]
# TODO: Implement export_markdown_report(summary, output_path) -> str

# Interview narrative checklist
checklist = [
    "Problem and hypothesis",
    "Method: stimuli + multi-agent loop",
    "Metrics and statistical evidence",
    "Optimization impact and limitations",
    "Next iteration plan"
]
for item in checklist:
    print("-", item)

## Reflection Prompt

What would you improve first if you had two additional weeks and why?